# Skrub + TabFM head

Follow-up to `../bakeoff/`. The bake-off fed one fixed MLP head with different
representations; **skrub `TableVectorizer` won** (test AUC 0.5982 ± 0.0014). This experiment
keeps the skrub representation and **replaces the MLP head with TabFM**.

[TabFM](https://github.com/google-research/tabfm) is Google's pretrained tabular foundation
model. It is not trained here: it learns *in context*. At prediction time it is shown the train
rows with their labels and predicts the test rows directly. Weights come from Hugging Face
`google/tabfm-1.0.0-pytorch` under the **non-commercial** `tabfm-non-commercial-v1.0` license
(the code is Apache-2.0).

Held identical to the bake-off, so the comparison is head vs head:
- **Sample:** the bake-off's fixed 50k `sample.parquet` (all years 2016–2024, stratified by year + label).
- **Split:** temporal by `FL_DAY`: train ≤ 9 / val 10–12 / test > 12. Never shuffled across days.
- **Target:** `|ARR_DELAY| > 15 min`.
- **Representation:** skrub `TableVectorizer(cardinality_threshold=40)` + `StandardScaler`, both fit on train only.
- **Metric:** test AUC (primary) + F1-macro + accuracy, mean ± std over seeds `[42, 43, 44]`.

The MLP is not re-run. Its numbers come from `../bakeoff/results.csv` (row `skrub`).

In [ ]:
# Colab: torch (CUDA), pandas, scikit-learn, pyarrow, pyyaml are preinstalled.
# TabFM comes from GitHub, pinned to a commit: the PyPI release (1.0.1) lacks
# cache_context, which encodes the context once instead of on every predict call.
# The uninstall drops a PyPI tabfm if one is present (same version number, so pip
# would otherwise keep it).
# After this cell, restart the session once (Runtime -> Restart session), then run from the top.
# Elsewhere: install a CUDA build of torch first, then these lines (see README).
!pip uninstall -y -q tabfm
!pip install -q "skrub>=0.10.0" "tabfm[pytorch] @ git+https://github.com/google-research/tabfm.git@fbb665569425fd2f490c6576b3af967876fe11ff"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)

device: cuda | torch 2.11.0+cu130


## Config

Edit the paths for the machine you run on. Data and representation settings mirror
`../bakeoff/config.yaml`, so the result is comparable with the bake-off.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ---- Paths ------------------------------------------------------------------
# The bake-off's fixed sample (written by ../bakeoff/prep/build_sample.py).
SAMPLE_PATH = Path("/content/drive/MyDrive/skrub_head_ablation/sample.parquet")
# Fallback only if SAMPLE_PATH is missing: folder holding Flight_tab_YYYY.csv +
# data_info_YYYY.yaml for every year below; the sample is then rebuilt identically.
TAB_DIR = None                      # e.g. Path("/content/drive/MyDrive/Tab")
OUT_DIR = Path("outputs")           # results, cached embedding

# ---- Data (identical to the bake-off) ---------------------------------------
DATA = dict(
    years=[2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
    sample_total=50000,
    seed=42,
    train_days_max=9,               # FL_DAY <= 9  -> train
    val_days_max=12,                # 9 < FL_DAY <= 12 -> val ; > 12 -> test
    delay_threshold=15.0,           # |ARR_DELAY| > 15 -> label 1
    exclude_cols=["FLIGHTS", "FL_YEAR", "FL_MONTH"],
)
# Schema from Datasets/Flight_tab.py (identical across years). DEP_DELAY / ARR_DELAY are targets.
_CAT_ALL = ["OP_CARRIER", "OP_CARRIER_FL_NUM", "FL_YEAR", "FL_MONTH", "FL_DAY", "FL_WEEK",
            "ORIGIN_INDEX", "DEST_INDEX"]
_CONT_ALL = ["CRS_DEP_TIME_MIN", "CRS_ARR_TIME_MIN", "CRS_ELAPSED_TIME", "FLIGHTS",
             "O_TEMP", "O_PRCP", "O_WSPD", "D_TEMP", "D_PRCP", "D_WSPD",
             "O_LATITUDE", "O_LONGITUDE", "D_LATITUDE", "D_LONGITUDE"]
CAT = [c for c in _CAT_ALL if c not in DATA["exclude_cols"]]
CONT = [c for c in _CONT_ALL if c not in DATA["exclude_cols"]]

# ---- Representation (kept from the bake-off) --------------------------------
SKRUB = dict(cardinality_threshold=40)
REUSE_EMBEDDING = True              # load OUT_DIR/emb_skrub.npy if it matches the sample

# ---- TabFM head -------------------------------------------------------------
SEEDS = [42, 43, 44]
TABFM = dict(
    n_estimators=32,       # ensemble members (library default). Lower to 8 if too slow / OOM.
    max_num_rows=None,     # context rows per member; None = all ~14.7k train rows.
                           # Set e.g. 8000 if the GPU runs out of memory.
    cache_context=True,    # encode the context once at fit, reuse it for every test chunk
    predict_chunk=4096,    # test rows per predict_proba call (bounds memory on 30k rows)
)
FORCE_RERUN = False        # True -> recompute seeds already in results_per_seed.csv

# ---- Reference: bake-off skrub + MLP head (../bakeoff/results.csv) ----
MLP_REFERENCE = dict(auc_mean=0.598228, auc_std=0.001408, f1_mean=0.537120, f1_std=0.005974,
                     acc_mean=0.600473, acc_std=0.001421)

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"{len(CAT)} categorical + {len(CONT)} continuous features | seeds {SEEDS}")

6 categorical + 13 continuous features | seeds [42, 43, 44]


## Load the fixed sample

Reads `sample.parquet`. If it is missing and `TAB_DIR` is set, the sample is rebuilt with the
bake-off's exact procedure (`prep/build_sample.py`: per-year quota proportional to size,
label-stratified, `random_state=42`) and cached to `OUT_DIR/sample.parquet`.

In [ ]:
def build_sample(tab_dir: Path, data: dict) -> pd.DataFrame:
    import yaml
    from sklearn.model_selection import train_test_split

    counts = {}
    for y in data["years"]:
        with open(tab_dir / f"data_info_{y}.yaml") as f:
            counts[y] = int(yaml.safe_load(f)["data_summary"]["shape"][0])

    # largest-remainder allocation of sample_total, proportional to year size
    total, grand = data["sample_total"], sum(counts.values())
    exact = {y: total * c / grand for y, c in counts.items()}
    quota = {y: int(np.floor(v)) for y, v in exact.items()}
    for y in sorted(exact, key=lambda y: exact[y] - quota[y], reverse=True)[: total - sum(quota.values())]:
        quota[y] += 1

    parts = []
    for y in data["years"]:
        if quota[y] <= 0:
            continue
        d = pd.read_csv(tab_dir / f"Flight_tab_{y}.csv").dropna()
        d["label"] = (d["ARR_DELAY"].abs() > data["delay_threshold"]).astype(int)
        d["FL_YEAR"] = y
        sub, _ = train_test_split(d, train_size=min(quota[y], len(d)),
                                  stratify=d["label"], random_state=data["seed"])
        parts.append(sub)
        print(f"  [{y}] sampled {len(sub)}")
        del d

    s = pd.concat(parts, ignore_index=True)
    s["split"] = np.select([s["FL_DAY"] <= data["train_days_max"],
                            s["FL_DAY"] <= data["val_days_max"]], ["train", "val"], default="test")
    return s


if SAMPLE_PATH.exists():
    df = pd.read_parquet(SAMPLE_PATH)
elif (OUT_DIR / "sample.parquet").exists():
    df = pd.read_parquet(OUT_DIR / "sample.parquet")
elif TAB_DIR is not None:
    df = build_sample(Path(TAB_DIR), DATA)
    df.to_parquet(OUT_DIR / "sample.parquet", index=False)
else:
    raise FileNotFoundError(f"{SAMPLE_PATH} not found and TAB_DIR is None — set one of them.")

missing = [c for c in CAT + CONT + ["label", "split"] if c not in df.columns]
assert not missing, f"sample is missing columns: {missing}"

tr_m = (df["split"] == "train").to_numpy()
va_m = (df["split"] == "val").to_numpy()
te_m = (df["split"] == "test").to_numpy()
y = df["label"].to_numpy().astype(int)
y_tr, y_va, y_te = y[tr_m], y[va_m], y[te_m]

print(f"rows: {len(df)} | train/val/test = {tr_m.sum()}/{va_m.sum()}/{te_m.sum()} | "
      f"delayed frac = {y.mean():.3f}   (bake-off: 14690/4985/30325, 0.408)")

rows: 50000 | train/val/test = 14690/4985/30325 | delayed frac = 0.408   (bake-off: 14690/4985/30325, 0.408)


## Skrub representation

Identical to `../bakeoff/embeddings/embed_skrub.py`: categoricals cast to `str`,
`TableVectorizer` fit on **train rows only**, then transformed for all rows. A `StandardScaler`
fit on train is applied on top, as the bake-off's MLP head did. The matrix is cached to
`OUT_DIR/emb_skrub.npy`.

In [ ]:
emb_path = OUT_DIR / "emb_skrub.npy"
Z = np.load(emb_path) if (REUSE_EMBEDDING and emb_path.exists()) else None
if Z is not None and Z.shape[0] != len(df):
    print(f"cached embedding has {Z.shape[0]} rows, sample has {len(df)} -> recomputing")
    Z = None

if Z is None:
    from skrub import TableVectorizer

    X = df[CAT + CONT].copy()
    for c in CAT:
        X[c] = X[c].astype(str)
    tv = TableVectorizer(cardinality_threshold=SKRUB["cardinality_threshold"])
    tv.fit(X[tr_m])
    Z = tv.transform(X)
    Z = np.asarray(getattr(Z, "values", Z), dtype=np.float32)
    np.save(emb_path, Z)

scaler = StandardScaler().fit(Z[tr_m])
cols = [f"z{i}" for i in range(Z.shape[1])]
# All-numeric DataFrames -> TabFM treats every skrub column as continuous.
X_tr, X_va, X_te = (pd.DataFrame(scaler.transform(Z[m]).astype(np.float32), columns=cols)
                    for m in (tr_m, va_m, te_m))
print(f"skrub dim = {Z.shape[1]}  (bake-off: 95)")

skrub dim = 95  (bake-off: 95)


## TabFM head

`fit` does not train anything. It stores the train rows as the context and prepares TabFM's
own preprocessing. Each of the `n_estimators` ensemble members sees a differently shuffled and
normalised view of the context, and their predictions are averaged. `random_state` = seed
controls that ensemble randomness. Val rows are only used to *report* a val AUC; nothing is
selected on them.

In [ ]:
_TABFM_MODEL = None


def load_tabfm():
    # Downloaded once from Hugging Face (google/tabfm-1.0.0-pytorch); runs in bf16 by design.
    global _TABFM_MODEL
    if _TABFM_MODEL is None:
        from tabfm import tabfm_v1_0_0_pytorch
        _TABFM_MODEL = tabfm_v1_0_0_pytorch.load(model_type="classification", device=str(DEVICE))
    return _TABFM_MODEL


def safe_auc(y_true, p):
    return roc_auc_score(y_true, p) if len(np.unique(y_true)) > 1 else float("nan")


def tabfm_kwargs(seed):
    # cache_context only exists in the GitHub build of tabfm, not the PyPI 1.0.1 release.
    import inspect
    from tabfm import TabFMClassifier

    kw = dict(n_estimators=TABFM["n_estimators"], max_num_rows=TABFM["max_num_rows"],
              cache_context=TABFM["cache_context"], random_state=seed)
    accepted = inspect.signature(TabFMClassifier.__init__).parameters
    dropped = [k for k in kw if k not in accepted]
    if dropped:
        print(f"  ! installed tabfm does not support {dropped}; running without "
              f"(slower: the context is re-encoded on every predict call)")
    return {k: v for k, v in kw.items() if k in accepted}


def run_tabfm(seed):
    from tabfm import TabFMClassifier

    t0 = time.time()
    clf = TabFMClassifier(model=load_tabfm(), **tabfm_kwargs(seed))
    clf.fit(X_tr, y_tr)
    pos = list(clf.classes_).index(1)

    def proba(X):
        bs = TABFM["predict_chunk"]
        return np.concatenate([clf.predict_proba(X.iloc[i:i + bs])[:, pos]
                               for i in range(0, len(X), bs)])

    val_auc = safe_auc(y_va, proba(X_va))
    p = proba(X_te)
    pred = (p > 0.5).astype(int)
    return dict(seed=seed, auc=safe_auc(y_te, p), f1=f1_score(y_te, pred, average="macro"),
                acc=accuracy_score(y_te, pred), val_auc=val_auc, time_s=time.time() - t0)

## Run

One TabFM pass per seed. Each seed is appended to `OUT_DIR/results_per_seed.csv` as soon as it
finishes, so a Colab disconnect only loses the seed in progress. Re-run the cell to continue.

In [ ]:
per_seed_csv = OUT_DIR / "results_per_seed.csv"
per_seed = pd.read_csv(per_seed_csv) if per_seed_csv.exists() else pd.DataFrame()

for seed in SEEDS:
    if not per_seed.empty and seed in per_seed["seed"].values:
        if not FORCE_RERUN:
            print(f"seed={seed}: already done, skipping")
            continue
        per_seed = per_seed[per_seed["seed"] != seed]
    m = run_tabfm(seed)
    print(f"seed={seed}: AUC={m['auc']:.4f} F1m={m['f1']:.4f} ACC={m['acc']:.4f} "
          f"| val AUC={m['val_auc']:.4f} | {m['time_s']:.0f}s")
    per_seed = pd.concat([per_seed, pd.DataFrame([m])], ignore_index=True)
    per_seed.to_csv(per_seed_csv, index=False)

per_seed

  ! installed tabfm does not support ['cache_context']; running without (slower: the context is re-encoded on every predict call)
seed=42: AUC=0.6337 F1m=0.5450 ACC=0.6170 | val AUC=0.6251 | 1291s
  ! installed tabfm does not support ['cache_context']; running without (slower: the context is re-encoded on every predict call)
seed=43: AUC=0.6335 F1m=0.5457 ACC=0.6168 | val AUC=0.6242 | 1289s
  ! installed tabfm does not support ['cache_context']; running without (slower: the context is re-encoded on every predict call)
seed=44: AUC=0.6335 F1m=0.5447 ACC=0.6165 | val AUC=0.6245 | 1290s


,seed,auc,f1,acc,val_auc,time_s
0,42,0.633661,0.545049,0.617016,0.625084,1290.764219
1,43,0.633545,0.545696,0.616752,0.624215,1289.243286
2,44,0.633537,0.544678,0.616488,0.624532,1289.611941


## Result vs the bake-off MLP head

Both rows use the same sample, split, skrub matrix and seeds; only the head differs. `σ` is
the AUC gap divided by the pooled seed std, a rough guide with 3 seeds (|σ| < ~2 is within noise).

In [ ]:
r = pd.read_csv(per_seed_csv)
r = r[r["seed"].isin(SEEDS)]
tabfm_row = dict(
    Head="tabfm", dim=Z.shape[1],
    auc_mean=r["auc"].mean(), auc_std=r["auc"].std(ddof=0),
    f1_mean=r["f1"].mean(), f1_std=r["f1"].std(ddof=0),
    acc_mean=r["acc"].mean(), acc_std=r["acc"].std(ddof=0),
    val_auc_mean=r["val_auc"].mean(), time_s_mean=r["time_s"].mean(),
    n_seeds=len(r), params=json.dumps(TABFM),
)
pd.DataFrame([tabfm_row]).to_csv(OUT_DIR / "results.csv", index=False)

ref = MLP_REFERENCE
d_auc = tabfm_row["auc_mean"] - ref["auc_mean"]
pooled = np.sqrt((tabfm_row["auc_std"] ** 2 + ref["auc_std"] ** 2) / 2)
print(f"Δ AUC (TabFM − MLP) = {d_auc:+.4f}" + (f"  ({d_auc / pooled:+.1f}σ)" if pooled > 0 else ""))

pd.DataFrame([
    {"Head": "mlp (bake-off)", **{k: ref[k] for k in ref}},
    {"Head": "tabfm", **{k: tabfm_row[k] for k in ref}},
]).set_index("Head").style.format("{:.4f}")

Δ AUC (TabFM − MLP) = +0.0354  (+35.5σ)


,auc_mean,auc_std,f1_mean,f1_std,acc_mean,acc_std
Head,,,,,,
mlp (bake-off),0.5982,0.0014,0.5371,0.0060,0.6005,0.0014
tabfm,0.6336,0.0001,0.5451,0.0004,0.6168,0.0002
